In [ ]:
import numpy as np
from scipy.special import logit, expit
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# below code is function to obtain random subsample from validation data set to get class balance equivalent to 
# that observed in a population. Although not ideal solution as throws away data to try imitate a population 
# rate that cannot verify

# def random_subsample_data(X_validate, y_validate, pop_rate, random_seed=42):
#     """ get random subsample of data, with proportion of positive class equal to 
#         specified population proportion.

#          X_validate: ndarray containing features
#          y_validate: 1D array containing target values
#          pop_rate: proportion of population that is expected to be positive class
#          random_seed: random seed for the random downsampling of the positive class """
    
#     # combine features and target back into one array
#     Xy = np.column_stack((X_validate, y_validate))


#     # get cases where target is positive class - note that y is final column!
#     Xy_pos = Xy[Xy[:, -1] == 1, :]
#     num_pos_sample = Xy_pos.shape[0]

#     # and where negative
#     Xy_neg = Xy[Xy[:, -1] == 0, :]
#     num_neg = Xy_neg.shape[0]

#     # get number of positive cases that will give specified population rate, while
#     # maintaining all negative classifications (rounded up)
#     num_pos_new = int(np.ceil(pop_rate*num_neg/(1 - pop_rate)))

#     # from positive cases, get random indicies to extract, with total number equivalent to
#     # first create random number generator with seed
#     rng = np.random.default_rng(seed=random_seed)

#     # then create the array using number generator method.
#     # ensures that picks unique rows rather than sampling with replacement as when using rng.integers
#     pos_index = rng.choice(num_pos_sample, size=num_pos_new, replace=False)
#     # pos_index = rng.integers(low=0, high=num_pos_sample, size=num_pos_new)

#     # use indexes to downsample positive cases
#     Xy_pos_downsampled = Xy_pos[pos_index, :]

#     # and then concatentate with negative cases to get full dataset again
#     Xy_new = np.vstack((Xy_pos_downsampled, Xy_neg))

#     # return columns seperately as features and target

#     #TODO: add tests for the downsampling function, and a warning if rate deviates from inserted 
#     # population rate more than some threshold (10%?)

#     return Xy_new[:, :-1], Xy_new[:, -1]


#  # downsample the validation data to obtain a proportion of positive classes equuivalent to that 
#     # specified in population (from config file) - to use in calibration of probabilities
#     # during validation
#     X_downsampled_fromV, y_downsampled_fromV = random_subsample_data(X_validate, y_validate, 
#                                                                      pop_rate=population_pos_class_rate, random_seed=random_seed)


# # create an initial empty JSON file for the inference times of all models
# Path(LOGS_PATH / 'data_v1').touch()

# # add braces so that can append to file
# with open(LOGS_PATH / 'data_v1' / 'data_balance.json', 'w') as f:
#     json.dump(sample_rate_dict, f)


# with open(DATA_PATH / 'processed' / 'X_validate_ds.pkl', 'wb') as f:
#     pkl.dump(X_downsampled_fromV, f)

# with open(DATA_PATH / 'processed' / 'y_validate_ds.pkl', 'wb') as f:
#     pkl.dump(y_downsampled_fromV, f)

In [ ]:
# this function adjust prediected probabilities of posiitive classification to account for 
# difference in class balance between the sample and population

# then also to plot the calibration curve alongside the confindence intervals for the proportions
def apply_prior_correction_logit(p, samp_rate, pop_rate):
    """ adjusts predicted probabilty (p) given difference in 
    proportion of class within sample (samp_rate) vs the population (pop_rate), 
    to account for oversampling of data. 
    Here using log form which better deals with probabilities very near 0 and 1 """

    delta = np.log(pop_rate/(1-pop_rate)) - np.log(samp_rate/(1-samp_rate))

    return expit(logit(p) + delta)

print(apply_prior_correction_logit(0.95, samp_rate=0.5, pop_rate=0.05))

# ##########
# # apply prior corrections to account for how the training data was oversampled from the population
# pred_proba_default_adj = apply_prior_correction_logit(p=pred_proba_default, 
#                                                 samp_rate=sample_pos_class_rate, 
#                                                 pop_rate=population_pos_class_rate)

# print('mean probability: ', pred_proba_default.mean())
# print('mean probability adjusted: ', pred_proba_default_adj.mean())

# plot distribution of predicted probalities
        
    
# # create df from y validate and corresponding predicted probability
# df_y_proba = pd.DataFrame(data=np.column_stack((y_validate_ds, pred_proba_default_adj)), columns=['label', 'pred_proba'])

# # create column for bin - using quantiles to ensure that equaL size. 
# # df_y_proba['bin'] = pd.qcut(df_y_proba['pred_proba'], q=200)

# # instead using sklearn default of equal width bins for intepretability of plot
# # set limits of bins to that of prediected probabilities (otherwise may get misleading visuals)
# df_y_proba['bin'] = pd.cut(df_y_proba['pred_proba'], bins=np.linspace(0, df_y_proba['pred_proba'].max(),5))
# # print(df_y_proba.columns)
# # in each bin number of actual positives in bin (can just use mean as binary problem)
# df_proportion_actual_positives_by_bin = df_y_proba.groupby('bin')['label'].mean()

# # and compute average prediected probability
# df_avg_pred_proba_by_bin = df_y_proba.groupby(['bin'])['pred_proba'].mean()

# ########
# # get confidence intervals for predicted probability
# ########

# # first need to get num samples in each bin
# count_each_bin = df_y_proba.groupby(['bin'])['bin'].count()

# # get standard error
# std_error = np.sqrt(1/(count_each_bin*df_proportion_actual_positives_by_bin*(1 - df_proportion_actual_positives_by_bin)))

# # get upper and lower 95% confidence intervals for each bin
# # perform element wise min and max so that clip intervals beyond 1 and 0
# ci_upper = np.minimum(df_proportion_actual_positives_by_bin + 1.96*std_error, np.array([1]))
# ci_lower = np.maximum(df_proportion_actual_positives_by_bin - 1.96*std_error, np.array([0]))

# # get wilson confidence intervals for small sample
# # lower, upper = proportion_confint(count=count_each_bin, nobs=)
# # linspace of same size as prediction arrays to get 'perfect calibration' plot
# linspace = np.linspace(0, 1, len(df_avg_pred_proba_by_bin))

# # now plot actual posistives vs the prediected probability
# plt.plot(df_avg_pred_proba_by_bin, df_proportion_actual_positives_by_bin, label='from_data')
# # plot 'perfect calibration curve'
# plt.plot(linspace, linspace, label='perfect')
# # plt.fill_between(df_avg_pred_proba_by_bin, df_proportion_actual_positives_by_bin, ci_upper, color='orange')
# # plt.fill_between(df_avg_pred_proba_by_bin, ci_lower, df_proportion_actual_positives_by_bin, color='orange')
# plt.xlabel('Avg prediected probability')
# plt.ylabel('proportion actual positives')
# plt.legend()
# plt.title(f'calibration curve using {model_name}')
# # plt.savefig(f'../plots/reliability_{model_name}')
# plt.show()


In [ ]:
# code for plotting the calibration curves and 
# reliability diagrams (could put into seperate plotting script)

####
# # Obtain probability and test calibration
# #############

# # make plot directory if no exist
# os.makedirs(BASE_PATH / 'plots', exist_ok=True)

# sns.histplot(pred_proba_default, bins=100)
# plt.xlabel('predicted probability')
# plt.ylabel('count')
# plt.title(f'{model_name}')
# plt.savefig(BASE_PATH / 'plots' / f'pred_prob_hist_{model_name}_noAdj')
# plt.close()

# sns.histplot(pred_proba_default, bins=100)
# plt.xlabel('predicted probability')
# plt.ylabel('count')
# plt.title(f'{model_name}' + f' brier score = {brier_score}')
# plt.savefig(BASE_PATH / 'plots' / f'pred_prob_hist_{model_name}_adj')
# plt.close()
# # plt.show()

# # display calibration curve
# CalibrationDisplay.from_estimator(X=X_validate, y=y_validate, estimator=model, n_bins=100)
# plt.savefig(BASE_PATH / 'plots' / f'reliability_{model_name}')
# plt.close()